# Q(V, φ) parametrization

Collected charge **Q** of ETL LGADs vs bias **V** and fluence **φ**. Data is read directly from `parametrization_data.xlsx` (one sheet per production).

Modules: `config.py` (settings) · `data_io.py` (Excel reader) · `model.py` (fits) · `plots.py` (ROOT plots).

**Model.** The analytic description of each production is the single smooth surface

$$Q(V,\varphi) = Q_\text{FLOOR} + \exp\big[(a_0 + a_1\varphi) + (b_0 + b_1\varphi)\,V\big]$$

fitted to all pooled points at once (`model.fit_global`). `Q_FLOOR = 0.55 fC` is the no-gain (bulk) charge the curve tends to at high fluence. The per-fluence fits `Q(V)=p0+p1·V+exp(p2+p3·V)` are kept as diagnostics (parameter tables). Charge error is `config.ERR_Q` (default 1 fC, switchable to 2).

Kernel: **Python (tb_analysis2)**. Full method: `docs/method.pdf`.

In [ ]:
import ROOT
%jsroot on
import importlib
import config, data_io, model, plots
for m in (config, data_io, model, plots):
    importlib.reload(m)

# charge error: default 1 fC, switchable to 2
config.set_charge_error(1.0)

sensors = data_io.load_sensors()
print(data_io.summary(sensors))
print('\ncharge error = +-%.1f fC | param_mode = %s' % (config.ERR_Q, config.PARAM_MODE))

In [ ]:
# fit every production (sheet)
# (1) per-fluence Q(V) fits -> parameters p0..p3 (diagnostics, see tables below)
results = model.fit_all(sensors, verbose=True)

# (2) THE analytic model: one smooth surface per production, fitted to ALL points
#     Q(V,phi) = Q_FLOOR + exp( a0 + a1*phi + (b0 + b1*phi)*V )
surfaces = model.fit_all_global(sensors)
print('\nAnalytic global law per production:')
for name, surf in surfaces.items():
    print('  ', name, '->', surf.law_string())

## Q(V) fits — pooled data + fit (open red = test beam, cross-check only)

In [ ]:
for name, r in results.items():
    plots.plot_QV(sensors, r).Draw()

## Parameters vs fluence (blue = law used; dashed grey = linear trend)

In [ ]:
for name, r in results.items():
    plots.plot_params(r).Draw()

## The analytic model vs data
The single analytic surface `Q(V,φ) = Q_FLOOR + exp(a0 + a1·φ + (b0 + b1·φ)·V)` overlaid on the pooled data at each measured fluence — one formula describing every fluence.

In [ ]:
for name, surf in surfaces.items():
    plots.plot_surface_vs_data(sensors, surf).Draw()

## Bias to reach the target charge, vs fluence

In [ ]:
plots.plot_bias_for_target(surfaces, Q_target=config.Q_TARGET, phi_hi=1.6e15).Draw()
print('Bias to reach Q = %.0f fC:' % config.Q_TARGET)
for name, surf in surfaces.items():
    row = {}
    for phi in surf.fluences:
        v = surf.V_for_Q(config.Q_TARGET, phi)
        row['%.1e' % phi] = 'n/a' if v != v else '%.0f V' % v
    print(' ', name, row)

## The analytic surface Q(V,φ) as a 2-D map

In [ ]:
tf2s = {name: surf.tf2() for name, surf in surfaces.items()}
c = ROOT.TCanvas('csurf', 'surfaces', 500 * len(tf2s), 450)
c.Divide(len(tf2s), 1)
for i, (name, f2) in enumerate(tf2s.items()):
    c.cd(i + 1); ROOT.gPad.SetRightMargin(0.16)
    f2.SetMaximum(20); f2.Draw('colz')
c.Draw()

## Operating point: charge obtainable & bias needed vs fluence (SEB-capped)

**Cell 1** — for each production: dashed = bias needed to reach the target charge (15 fC), capped at the SEB limit `V_SEB` (HPK 550 V, FBK 600 V); solid = charge actually obtainable = `min(target, Q(V_SEB, φ))` (flat at the target until the SEB knee, then the charge available at `V_SEB`).

**Cell 2** — fix a charge (e.g. 8 fC) and read off the bias needed at each fluence (flagged if it exceeds `V_SEB`).

In [ ]:
# === CELL 1 — charge obtained & bias needed vs fluence, with the SEB cap ===
# Uses the analytic global surface (computed above): one smooth trend, charge
# converges to the no-gain floor config.Q_FLOOR (= 0.55 fC) at high fluence.
TARGET = config.PLATEAU_CHARGE            # 15 fC (solid-line plateau)
# V_SEB per production from config.V_SEB (HPK 550 V, FBK 600 V); override with vmax={...}
plots.plot_charge_bias_vs_fluence(surfaces, target=TARGET, vmax=None).Draw()

In [ ]:
# === CELL 2 — fix a charge, get the bias needed at each fluence ===
Q_FIX = 8.0   # fC  (change me)

plots.plot_bias_for_target(surfaces, Q_target=Q_FIX, phi_hi=1.6e15).Draw()

print('Bias needed to reach Q = %.1f fC (global surface):' % Q_FIX)
for name, surf in surfaces.items():
    vmax = plots._vmax_for(name, None)
    for phi in surf.fluences:
        v = surf.V_for_Q(Q_FIX, phi, v_hi=1200)
        if v != v:
            s = 'not reachable'
        else:
            s = '%.0f V' % v + ('  (> V_SEB = %.0f V!)' % vmax if v > vmax else '')
        print('  %-6s  phi=%.2e  ->  %s' % (name, phi, s))

## Parameters of the production
Per-fluence Q(V) fit parameters `p0..p3` (with errors and χ²/ndf) and the vs-fluence laws actually used. Also written to `parameters_<prod>.csv`.

In [ ]:
# === Parameters of each production ===
# (1) per-fluence Q(V) fit parameters p0..p3 (+ errors, chi2/ndf)
# (2) the analytic global law  Q(V,phi) = Q_FLOOR + exp(a0 + a1*phi + (b0+b1*phi)*V)
import pandas as pd
for name, r in results.items():
    print('=' * 72)
    print(name)
    df = model.param_table(r)
    display(df.round(4))
    surf = surfaces[name]
    print('analytic global law:')
    print('   ', surf.law_string())
    print('   coefficients  a0=%.5e  a1=%.5e  b0=%.5e  b1=%.5e  (Q_FLOOR=%.3g fC)'
          % (surf.popt[0], surf.popt[1], surf.popt[2], surf.popt[3], surf.qfloor))
    df.to_csv('parameters_%s.csv' % model._short(name), index=False)
    print('-> saved parameters_%s.csv' % model._short(name))